In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from snntorch import spikegen
from customdataset import canny
import matplotlib.pyplot as plt
import shutil
import math



resize = transforms.Resize((320, 320))
to_tensor = transforms.ToTensor()

def spike_from_image(img_path, timestep=4):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    edges = canny(img)

    # Igual que en el dataset
    img = Image.fromarray(edges)
    img = resize(img)
    img = to_tensor(img)
    spikes = spikegen.rate(img, num_steps=timestep)

    return spikes

def generate_spikes(img_dir, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    spike_trains = {}

    for f in os.listdir(img_dir):
        if f.endswith(".jpg"):
            spikes = spike_from_image(os.path.join(img_dir, f))
            spike_img = (spikes.sum(dim=0)[0] > 0).numpy().astype(np.uint8) * 255
            cv2.imwrite(os.path.join(save_dir, f.replace(".jpg", "_spike.png")), spike_img)
            spike_trains[f] = spikes

    return spike_trains

def detect_noisy(spike_trains, threshold=0.35):
    return [
        fname
        for fname, spike_map in spike_trains.items()
        if (spike_map.sum() / spike_map.numel()) >= threshold
    ]



def move_noisy_spikes(noisy_list, pre_dir, noisy_dir):
    os.makedirs(noisy_dir, exist_ok=True)

    for fname in noisy_list:
        spike = fname.replace(".jpg", "_spike.png")
        shutil.move(
            os.path.join(pre_dir, spike),
            os.path.join(noisy_dir, spike)
        )

def build_clean_dataset(img_dir, lbl_dir, clean_img_dir, clean_lbl_dir, noisy_list):
    noisy_set = set(noisy_list)

    shutil.rmtree(clean_img_dir)
    shutil.rmtree(clean_lbl_dir)

    os.makedirs(clean_img_dir)
    os.makedirs(clean_lbl_dir)

    for fname in os.listdir(img_dir):
        if fname not in noisy_set:
            shutil.copy(os.path.join(img_dir, fname),
                        os.path.join(clean_img_dir, fname))

    for fname in os.listdir(lbl_dir):
        if fname.replace(".txt", ".jpg") not in noisy_set:
            shutil.copy(os.path.join(lbl_dir, fname),
                        os.path.join(clean_lbl_dir, fname))
                        
                        
                        


def show_noisy_images(noisy_list, noisy_dir, n=50, cols=3):
    subset = noisy_list[:n]
    rows = math.ceil(len(subset) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 3))
    axes = axes.flatten()

    for ax, fname in zip(axes, subset):
        spike_img = cv2.imread(os.path.join(noisy_dir, fname.replace(".jpg", "_spike.png")),
                               cv2.IMREAD_GRAYSCALE)
        ax.imshow(spike_img, cmap="gray")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def show_clean_and_spike(clean_dir, spike_dir, n=10):
    clean_files = [f for f in os.listdir(clean_dir) if f.endswith(".jpg")]
    clean_files = clean_files[:n]

    for fname in clean_files:
        clean_path = os.path.join(clean_dir, fname)
        spike_path = os.path.join(spike_dir, fname.replace(".jpg", "_spike.png"))

        clean_img = cv2.cvtColor(cv2.imread(clean_path), cv2.COLOR_BGR2RGB)
        spike_img = cv2.imread(spike_path, cv2.IMREAD_GRAYSCALE)

        plt.figure(figsize=(8, 4))

        plt.subplot(1, 2, 1)
        plt.imshow(clean_img)
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(spike_img, cmap="gray")
        plt.axis("off")

        plt.show()